# Notebook 1 - Data understanding

> **Mentor briefing** - the seven questions we answer for every phase.

| Question | Answer |
|---|---|
| **1. What are we doing?** | Load the Online Retail II file, look at it from every angle and write down what is wrong with it. |
| **2. Why?** | A model can only be as good as its data. Ten minutes of inspection saves days of debugging a model that learned from garbage. |
| **3. Real-world problem it solves** | A retailer exports every sale line to a spreadsheet. Before anyone can report revenue or build recommendations, someone must know what each column means and which rows can be trusted. |
| **4. What does the code do?** | `load_raw_transactions` reads the file and standardises column names; `profile_dataframe` and a few pandas one-liners measure rows, types, missing values and data-quality problems. |
| **5. Expected output** | Roughly one million rows x 8 columns, about a quarter of them without a CustomerID, plus cancellations, free items, service codes and duplicates. |
| **6. What can go wrong?** | Excel stores dates and codes inconsistently (a StockCode like `85123A` can be turned into a number); wrong encodings; two sheets that must be combined; silently treating guest checkouts as one huge customer. |
| **7. How do professionals approach this?** | They profile first, decide cleaning rules second, and write every decision down (see `docs/dataset_documentation.md`). They never clean by guessing. |


## Concepts you need first

* **Dataset** - a table of recorded facts. Here: one row per *product line on an invoice*.
* **Observation** (a row, also called a record or sample) - one thing we measured. Here: "invoice 489434 contained 12 units of product 85048".
* **Feature** (a column, also called a variable) - one property of every observation: `Quantity`, `UnitPrice`, `Country` ...
* **EDA (Exploratory Data Analysis)** - looking at the data with summaries and charts *before* modelling, to learn its shape, spot problems and form questions.
* **Missing data** - cells with no value (`NaN`). Missing is a *message*: a missing CustomerID here means a guest checkout, not a typo.
* **Why inspect before ML?** Models do not know the data is broken. They will happily learn that a 80,995-unit order is normal.

### Where the data comes from
UCI Machine Learning Repository, *Online Retail II*: transactions of a UK-based online gift shop, **1 Dec 2009 - 9 Dec 2011**. Official page: <https://archive.ics.uci.edu/dataset/502/online+retail+ii>.

In [1]:
import sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))                      # lets us import our own code from src/
from src.utils.config import get_paths, RANDOM_STATE
paths = get_paths(ROOT)

sns.set_theme(style="whitegrid", context="notebook", palette=["#2563eb", "#0d9488", "#4f46e5", "#eda100", "#eb6834", "#e87ba4"])
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "figure.figsize": (9, 4.2)})
pd.options.display.float_format = "{:,.2f}".format
pd.options.display.width = 140

## 1. Load the file

In [2]:
from src.data.load_data import load_raw_transactions, DATASET_URL
raw, source = load_raw_transactions(paths, allow_synthetic=False)
print("data source :", source)
print("shape       :", raw.shape, "(rows, columns)")
raw.head()

data source : uci_online_retail_ii
shape       : (1067371, 8) (rows, columns)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12.00,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12.00,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12.00,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48.00,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24.00,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom


Our loader accepts the official `online_retail_II.xlsx` (two sheets), a CSV export, or the older *Online Retail* file, and renames columns so everything downstream sees the same eight names.

In [3]:
raw.dtypes.to_frame('dtype').T

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
dtype,str,str,str,float64,datetime64[us],float64,float64,str


### Column dictionary

| Column | Meaning | Type | Notes |
|---|---|---|---|
| `InvoiceNo` | Order number | text | A leading **C** means *cancellation / return* |
| `StockCode` | Product code | text | Some codes are not products (`POST`, `M`, `D`, `BANK CHARGES` ...) |
| `Description` | Product name | text | Inconsistent spelling/case; sometimes missing |
| `Quantity` | Units on this line | integer | **Negative** for returns and stock adjustments |
| `InvoiceDate` | When the order was placed | datetime | Used for every time-based step |
| `UnitPrice` | Price per unit, GBP | float | Zero for free items / adjustments |
| `CustomerID` | Customer identifier | float -> integer | **Missing** for guest checkouts |
| `Country` | Customer country | text | 40+ countries, mostly United Kingdom |

## 2. Missing values

In [4]:
missing = pd.DataFrame({"missing": raw.isna().sum(), "percent": raw.isna().mean() * 100}).sort_values("missing", ascending=False)
missing

,missing,percent
CustomerID,243007,22.77
Description,4382,0.41
StockCode,0,0.00
InvoiceNo,0,0.00
Quantity,0,0.00
InvoiceDate,0,0.00
UnitPrice,0,0.00
Country,0,0.00


`CustomerID` is the big one. We **cannot** build customer-level features for those rows - but they are still real sales, so we keep them for company-level revenue and product analysis (see Notebook 3).

In [5]:
from src.data.validate_data import check_schema, profile_dataframe, NON_PRODUCT_CODES
assert check_schema(raw) == [], check_schema(raw)       # fails loudly if a column is missing or InvoiceDate is not a datetime
print("schema OK")
raw.describe(include="all").T[["count", "unique", "mean", "min", "50%", "max"]]

schema OK


,count,unique,mean,min,50%,max
InvoiceNo,1067371,53628,NaN,NaN,NaN,NaN
StockCode,1067371,5304,NaN,NaN,NaN,NaN
Description,1062989,5655,NaN,NaN,NaN,NaN
Quantity,"1,067,371.00",NaN,9.94,"-80,995.00",3.00,"80,995.00"
InvoiceDate,1067371,NaN,2011-01-02 21:13:55.394029,2009-12-01 07:45:00,2010-12-07 15:28:00,2011-12-09 12:50:00
UnitPrice,"1,067,371.00",NaN,4.65,"-53,594.36",2.10,"38,970.00"
CustomerID,"824,364.00",NaN,"15,324.64","12,346.00","15,255.00","18,287.00"
Country,1067371,43,NaN,NaN,NaN,NaN


Look at `Quantity` and `UnitPrice`: the **minimum is negative** and the **maximum is enormous**. The mean and the median are far apart. That is the first sign of a *skewed* distribution with outliers (Notebook 2).

## 3. Hunt for data-quality problems
Each line below measures one suspicion.

In [6]:
inv = raw["InvoiceNo"].astype(str)
is_cancel = inv.str.upper().str.startswith("C")
problems = pd.Series({
    "exact duplicate rows": raw.duplicated().sum(),
    "cancellation invoices (InvoiceNo starts with C)": is_cancel.sum(),
    "negative quantity": (raw["Quantity"] < 0).sum(),
    "negative quantity WITHOUT a cancellation invoice": ((raw["Quantity"] < 0) & ~is_cancel).sum(),
    "zero quantity": (raw["Quantity"] == 0).sum(),
    "unit price == 0": (raw["UnitPrice"] == 0).sum(),
    "unit price < 0": (raw["UnitPrice"] < 0).sum(),
    "missing CustomerID (guest checkout)": raw["CustomerID"].isna().sum(),
    "missing Description": raw["Description"].isna().sum(),
    "service / non-product StockCode": raw["StockCode"].astype(str).str.upper().isin(NON_PRODUCT_CODES).sum(),
    "|Quantity| >= 10,000 (suspicious)": (raw["Quantity"].abs() >= 10_000).sum(),
}).rename("rows").to_frame()
problems["percent of all rows"] = problems["rows"] / len(raw) * 100
problems

,rows,percent of all rows
exact duplicate rows,34335,3.22
cancellation invoices (InvoiceNo starts with C),19494,1.83
negative quantity,22950,2.15
negative quantity WITHOUT a cancellation invoice,3457,0.32
zero quantity,0,0.00
unit price == 0,6202,0.58
unit price < 0,5,0.00
missing CustomerID (guest checkout),243007,22.77
missing Description,4382,0.41
service / non-product StockCode,5830,0.55


In [7]:
# Look at the evidence instead of trusting the counts.
print("A normal cancellation - customer returns goods:")
display(raw[is_cancel].head(3))
print("The same product being ordered AND cancelled in a huge quantity (a data-entry error and its correction):")
display(raw[raw["Quantity"].abs() >= 10_000].sort_values("InvoiceDate"))

A normal cancellation - customer returns goods:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12.00,2009-12-01 10:33:00,2.95,"16,321.00",Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6.00,2009-12-01 10:33:00,1.65,"16,321.00",Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4.00,2009-12-01 10:33:00,4.25,"16,321.00",Australia


The same product being ordered AND cancelled in a huge quantity (a data-entry error and its correction):


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,"19,152.00",2010-02-15 11:57:00,0.10,"13,902.00",Denmark
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,"12,960.00",2010-03-17 13:09:00,0.10,"13,902.00",Denmark
127167,501534,21092,SET/6 STRAWBERRY PAPER PLATES,"12,480.00",2010-03-17 13:09:00,0.10,"13,902.00",Denmark
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,"12,960.00",2010-03-17 13:09:00,0.10,"13,902.00",Denmark
127169,501534,21085,SET/6 WOODLAND PAPER CUPS,"12,744.00",2010-03-17 13:09:00,0.10,"13,902.00",Denmark
135027,502269,21984,PACK OF 12 PINK PAISLEY TISSUES,"10,000.00",2010-03-23 15:36:00,0.25,"17,940.00",United Kingdom
135028,502269,21982,PACK OF 12 SUKI TISSUES,"10,000.00",2010-03-23 15:36:00,0.25,"17,940.00",United Kingdom
135029,502269,21980,PACK OF 12 RED SPOTTY TISSUES,"10,000.00",2010-03-23 15:36:00,0.25,"17,940.00",United Kingdom
135030,502269,21981,PACK OF 12 WOODLAND TISSUES,"10,000.00",2010-03-23 15:36:00,0.25,"17,940.00",United Kingdom
192197,507637,84016,FLAG OF ST GEORGE CAR FLAG,"10,200.00",2010-05-10 14:55:00,0.00,NaN,United Kingdom


### What this tells us (decisions for Notebook 3 / `src/data/cleaning.py`)

| Finding | Decision |
|---|---|
| Duplicate rows | Remove - identical rows are double entries |
| Cancellations (`C...`, negative quantity) | **Keep as `return` records**, do not delete. Net revenue = sales - returns |
| Negative quantity without a `C` invoice | Remove - stock write-offs, not customer behaviour |
| Zero / negative price | Remove - free items and accounting adjustments are not sales |
| Service codes (`POST`, `D`, `M` ...) | Remove from product analysis - they are fees, not products |
| Quantity >= 10,000 or price >= 5,000 | Remove with their reversal - data-entry blow-ups that would distort every average |
| Missing CustomerID | Keep for company analysis, exclude from customer analysis |
| Messy descriptions | Upper-case, trim, one canonical name per StockCode |

In [8]:
print("Orders (unique invoices):", raw["InvoiceNo"].nunique())
print("Customers (known)       :", int(raw["CustomerID"].nunique()))
print("Products (StockCodes)   :", raw["StockCode"].nunique())
print("Countries               :", raw["Country"].nunique())
print("Period                  :", raw["InvoiceDate"].min(), "->", raw["InvoiceDate"].max())

Orders (unique invoices): 53628
Customers (known)       : 5942
Products (StockCodes)   : 5304
Countries               : 43
Period                  : 2009-12-01 07:45:00 -> 2011-12-09 12:50:00


## Check yourself
1. Why is *deleting every negative quantity* a mistake? (Hint: what happens to revenue?)
2. A customer appears with `CustomerID = NaN` in 22% of rows. Why can't we simply treat them as "customer NaN"?
3. Which columns would you expect to be **skewed**? Verify it in Notebook 2.

**Acceptance for Phase 1:** the dataset loads, its structure is documented (`docs/dataset_documentation.md`) and every data-quality problem has a named treatment. Tick.